In [1]:
"""
Audit: how much of NB3's linear interpolation is edge extrapolation
vs genuine interior interpolation?

Run from the project root (same directory as NB3).
Reads: intermediary/master_data_wide.csv
Applies the same sample filters and variable selection as NB3 Section 9-10.

Outputs a table per variable showing:
  - how many country-series have leading gaps (backward extrapolation)
  - how many have trailing gaps (forward extrapolation)
  - how many have only interior gaps
  - average gap length for each type
"""

import pandas as pd
import numpy as np

# ── Load and filter (mirrors NB3 Section 9) ──
df = pd.read_csv("intermediary/master_data_wide.csv")
if "Unnamed: 0" in df.columns:
    df.drop(columns="Unnamed: 0", inplace=True)

# NB3's omit list
omit_countries = [
    'BDI', 'BTN', 'CAF', 'ERI', 'FJI', 'GUY', 'ISL', 'MNE', 'NCL',
    'PRK', 'SLB', 'SLE', 'SSD', 'SUR', 'SYR', 'GUF', 'TWN', 'CUB',
    'TLS', 'AFG', 'TKM', 'KHM', 'XKX', 'BEN',
]
df = df[~df["Country Code"].isin(omit_countries)]

# NB3's keep_vars (only the ones that get interpolated)
EXCLUDE_VARS = [
    'Landlocked',
    'Subsoil_Metals_Dominant',
    'Hydrocarbons_Dominant',
    'Precious_Metals_Dominant',
    'Total_Reserves_Value',
    'Total_Reserves',
]

id_cols = ['Country Code', 'Country Name', 'Year']
data_cols = [c for c in df.columns if c not in id_cols and c not in EXCLUDE_VARS
             and c not in ['Total_Production', 'Total_Production_Value',
                           'Total_Reserves', 'Total_Reserves_Value',
                           'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
                           'Precious_Metals_Dominant', 'Landlocked']]

# Only keep numeric columns that actually exist
data_cols = [c for c in data_cols if c in df.columns and df[c].dtype in ['float64', 'int64']]

years = sorted(df['Year'].unique())
min_year, max_year = min(years), max(years)
n_years = len(years)

# ── Audit each variable x country pair ──
results = []

for var in data_cols:
    n_leading = 0
    n_trailing = 0
    n_interior_only = 0
    n_fully_missing = 0
    n_complete = 0
    leading_lengths = []
    trailing_lengths = []
    interior_lengths = []
    worst_leading = []

    for code in df['Country Code'].unique():
        series = (
            df[df['Country Code'] == code][['Year', var]]
            .sort_values('Year')
            .set_index('Year')
            .reindex(years)[var]
        )

        observed = series.dropna()
        if len(observed) == 0:
            n_fully_missing += 1
            continue
        if len(observed) == n_years:
            n_complete += 1
            continue

        first_obs_year = observed.index.min()
        last_obs_year = observed.index.max()

        has_leading = first_obs_year > min_year
        has_trailing = last_obs_year < max_year

        # Interior gaps: missing values between first and last observed
        interior_series = series.loc[first_obs_year:last_obs_year]
        n_interior_missing = interior_series.isna().sum()

        if has_leading:
            gap_len = first_obs_year - min_year
            n_leading += 1
            leading_lengths.append(gap_len)
            worst_leading.append((code, var, gap_len, first_obs_year))
        if has_trailing:
            gap_len = max_year - last_obs_year
            n_trailing += 1
            trailing_lengths.append(gap_len)
        if not has_leading and not has_trailing and n_interior_missing > 0:
            n_interior_only += 1
            interior_lengths.append(n_interior_missing)

    results.append({
        'Variable': var,
        'Countries': df['Country Code'].nunique(),
        'Complete': n_complete,
        'Fully Missing': n_fully_missing,
        'Leading Extrap': n_leading,
        'Avg Leading Yrs': round(np.mean(leading_lengths), 1) if leading_lengths else 0,
        'Max Leading Yrs': max(leading_lengths) if leading_lengths else 0,
        'Trailing Extrap': n_trailing,
        'Avg Trailing Yrs': round(np.mean(trailing_lengths), 1) if trailing_lengths else 0,
        'Max Trailing Yrs': max(trailing_lengths) if trailing_lengths else 0,
        'Interior Only': n_interior_only,
    })

    # Collect worst offenders
    if worst_leading:
        worst_leading.sort(key=lambda x: x[2], reverse=True)

results_df = pd.DataFrame(results).sort_values('Leading Extrap', ascending=False)

# ── Print report ──
print("=" * 100)
print("INTERPOLATION EDGE AUDIT")
print("=" * 100)
print(f"\nPanel: {min_year}-{max_year} ({n_years} years)")
print(f"Countries: {df['Country Code'].nunique()}")
print(f"Variables audited: {len(data_cols)}\n")

print("── SUMMARY BY VARIABLE ──\n")
print(results_df.to_string(index=False))

# Flag variables with significant edge extrapolation
print("\n\n── VARIABLES WITH >5 COUNTRIES EXTRAPOLATED >=5 YEARS ──\n")
flagged = results_df[
    (results_df['Leading Extrap'] > 5) & (results_df['Avg Leading Yrs'] >= 5)
    | (results_df['Trailing Extrap'] > 5) & (results_df['Avg Trailing Yrs'] >= 5)
]
if len(flagged) > 0:
    print(flagged[['Variable', 'Leading Extrap', 'Avg Leading Yrs',
                    'Max Leading Yrs', 'Trailing Extrap', 'Avg Trailing Yrs',
                    'Max Trailing Yrs']].to_string(index=False))
else:
    print("None found.")

# ── Worst individual cases (leading extrapolation > 10 years) ──
print("\n\n── WORST INDIVIDUAL CASES (leading gap > 10 years) ──\n")
all_worst = []
for var in data_cols:
    for code in df['Country Code'].unique():
        series = (
            df[df['Country Code'] == code][['Year', var]]
            .sort_values('Year')
            .set_index('Year')
            .reindex(years)[var]
        )
        observed = series.dropna()
        if len(observed) == 0:
            continue
        first_obs = observed.index.min()
        if first_obs - min_year > 10:
            all_worst.append({
                'Country': code,
                'Variable': var,
                'First Obs Year': first_obs,
                'Gap (years)': first_obs - min_year,
            })

if all_worst:
    worst_df = pd.DataFrame(all_worst).sort_values('Gap (years)', ascending=False)
    print(worst_df.head(30).to_string(index=False))
    print(f"\nTotal cases with >10yr leading extrapolation: {len(worst_df)}")
else:
    print("None found.")

# ── Total cells affected ──
total_leading_cells = sum(r['Leading Extrap'] * r['Avg Leading Yrs'] for _, r in results_df.iterrows())
total_trailing_cells = sum(r['Trailing Extrap'] * r['Avg Trailing Yrs'] for _, r in results_df.iterrows())
total_panel_cells = df['Country Code'].nunique() * n_years * len(data_cols)
total_missing = df[data_cols].isna().sum().sum()

print(f"\n\n── SCALE ──")
print(f"Total panel cells: {total_panel_cells:,.0f}")
print(f"Total missing cells (pre-imputation): {total_missing:,.0f} ({100*total_missing/total_panel_cells:.1f}%)")
print(f"Approx cells filled by leading extrapolation: {total_leading_cells:,.0f} ({100*total_leading_cells/total_panel_cells:.1f}%)")
print(f"Approx cells filled by trailing extrapolation: {total_trailing_cells:,.0f} ({100*total_trailing_cells/total_panel_cells:.1f}%)")
print(f"Approx cells filled by interior interpolation: {total_missing - total_leading_cells - total_trailing_cells:,.0f}")

FileNotFoundError: [Errno 2] No such file or directory: 'intermediary/master_data_wide.csv'